### Feature Engineering & Purchase Behavior Simulation

This module transforms the simulated customer dataset by creating
behavior-driven features required for hypothesis testing.

The process includes:
- Normalizing numerical features.
- Generating purchase probability based on customer behavior.
- Simulating purchase outcomes.
- Preparing the final dataset for A/B hypothesis testing.

In [2]:
# importing files
import os
import numpy as np
import pandas as pd

In [72]:
# path config 
data_path = os.path.join('../', 'data', 'stimulated_data.csv')
data_path_prob = os.path.join('../', 'data', 'stimulated_data_prob.csv')


In [8]:
# Loading data set 
data = pd.read_csv(data_path)

In [9]:
data.head()

,user_id,age,gender,country,device,session_duration,pages_viewed,previous_orders,traffic_source,experiment_group
0,1,61,Female,India,Mobile,3.45,5,1,Organic,A
1,2,38,Male,India,Mobile,1.04,2,4,Social,B
2,3,51,Female,India,Tablet,1.67,4,2,Email,B
3,4,24,Male,USA,Mobile,2.82,4,1,Organic,A
4,5,27,Male,UK,Tablet,5.00,2,4,Organic,B


In [48]:
# Feature Engineering Function 

# use min max sclar logic to stimulate weight for features and 
# to keep the total probability between 0 and 1
def min_max(col):
    return (col - col.min()) / (col.max() - col.min())

# probability calculated using genrated weights of features such that it could reflec the real relation 
BASE_PROB = 0.10

ORDER_WEIGHT = 0.35
PAGE_WEIGHT = 0.25
SESSION_WEIGHT = 0.20
GROUP_BONUS = 0.05

def calculate_purchase_probability(df):

    df['purchase_prob'] = (
        BASE_PROB
        + SESSION_WEIGHT * df['session_norm']
        + PAGE_WEIGHT * df['pages_norm']
        + ORDER_WEIGHT * df['orders_norm']
    )

    df['purchase_prob'] = np.where(
        df['experiment_group'] == 'B',
        df['purchase_prob'] + GROUP_BONUS,
        df['purchase_prob']
    )

    return df

In [49]:
data['session_norm'] = min_max(data['session_duration'])
data['pages_norm'] = min_max(data['pages_viewed'])
data['orders_norm'] = min_max(data['previous_orders'])

In [50]:
data.head()

,user_id,age,gender,country,device,session_duration,pages_viewed,previous_orders,traffic_source,experiment_group,session_norm,pages_norm,orders_norm
0,1,61,Female,India,Mobile,3.45,5,1,Organic,A,0.185369,0.20,0.111111
1,2,38,Male,India,Mobile,1.04,2,4,Social,B,0.035958,0.05,0.444444
2,3,51,Female,India,Tablet,1.67,4,2,Email,B,0.075015,0.15,0.222222
3,4,24,Male,USA,Mobile,2.82,4,1,Organic,A,0.146311,0.15,0.111111
4,5,27,Male,UK,Tablet,5.00,2,4,Organic,B,0.281463,0.05,0.444444


In [ ]:
# Generating purchase probability 
data = calculate_purchase_probability(data)

In [60]:
data['purchase_prob'].describe()

count    10000.000000
mean         0.265074
std          0.069048
min          0.105208
25%          0.215919
50%          0.258349
75%          0.307931
max          0.571201
Name: purchase_prob, dtype: float64

In [71]:
data.groupby('experiment_group')['purchase_prob'].agg(
    ['mean', 'median', 'std', 'min', 'max']
)

,mean,median,std,min,max
experiment_group,,,,,
A,0.239167,0.231780,0.064542,0.105208,0.571201
B,0.290508,0.283454,0.063667,0.152728,0.560437


- Observation :
    Group B have higher mean and median customers comared to Group A.

- Conclusion : 
    Customers in Group B are more likely to purchase the product.

- Reason : 
    A small intentional change was introduced for group b to simulate the impact of any new recomendation system to increase the customer likehood of purchase.

In [74]:
data.to_csv(data_path_prob, index=False)